# IndicTrans2 LoRA Fine-Tuning — Conversational Domain Adaptation

**Goal:** Fine-tune `ai4bharat/indictrans2-en-indic-dist-200M` with LoRA on **conversational / colloquial**
English–Hindi text, a domain AI4Bharat themselves flag as weaker for the base model
(see the dedicated `IN22-Conv` benchmark subset in the official IndicTrans2 release).

Unlike a generic-domain fine-tune (e.g. plain IITB), this targets a gap the base model is more likely
to actually have — informal register, contractions, dialogue-style phrasing — which gives LoRA
adaptation real room to improve over the base model, rather than fighting for marginal gains on
text the base model was already heavily pretrained on (Samanantar v2 / BPCC already swallow IITB-style
general text).

**Pipeline:**
1. Setup & config
2. Load conversational parallel data (OpenSubtitles en-hi)
3. Clean, dedupe, filter, split
4. Preprocess with `IndicProcessor` (labels use `is_target=True` — this is the fix for a label-corruption
   bug that silently collapses BLEU; see note in the preprocessing cell)
5. Tokenize & cache to Drive
6. Baseline (zero-shot) evaluation of the base model on held-out conversational data
7. LoRA fine-tune
8. Evaluate fine-tuned model, compare BLEU / chrF++ against base
9. Qualitative sample translations

> **Note on `is_target`:** `IndicProcessor.preprocess_batch(text, src_lang, tgt_lang, is_target=False)` —
> when `is_target=False` (the default), the function **prepends a language-tag prefix** to the text
> (needed for encoder inputs). If you preprocess **decoder labels** the same way, you bake a bogus
> tag sequence into every training target, which silently wrecks BLEU after fine-tuning. This notebook
> always passes `is_target=True` when preprocessing target/label text — do not remove it.


## 1. Setup

In [ ]:
# Mount Google Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!
!pip install -q "transformers==4.28.0" # Downgraded for IndicTransToolkit compatibility
!pip install -q IndicTransToolkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 17.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.4 MB/

In [ ]:
!pip install -q accelerate peft sacrebleu sentencepiece datasets
# Removed redundant IndicTransToolkit install
!pip install -q indic-nlp-library sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.9 MB/s eta 0:00:00


In [ ]:
import os
import json
import random
import re
import time

import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from IndicTransToolkit.processor import IndicProcessor
import sacrebleu

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


ImportError: cannot import name 'PreTrainedTokenizerBase' from 'transformers.tokenization_utils' (unknown location)

In [ ]:
# ---- Config ----
PROJECT_DIR = "/content/drive/MyDrive/NMT_Project_Conv"
os.makedirs(PROJECT_DIR, exist_ok=True)

BASE_MODEL = "ai4bharat/indictrans2-en-indic-dist-200M"
SRC_LANG = "eng_Latn"
TGT_LANG = "hin_Deva"

MAX_TOKEN_LENGTH = 128

# Dataset sizing — adjust down for CPU, up for GPU
TRAIN_SIZE = 50000      # set smaller (e.g. 2000) for a CPU smoke test
VAL_SIZE = 1000

TOKENIZED_TRAIN_PATH = os.path.join(PROJECT_DIR, "tokenized_train_conv")
TOKENIZED_VAL_PATH = os.path.join(PROJECT_DIR, "tokenized_val_conv")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "indictrans2_lora_conv")

print("Project dir:", PROJECT_DIR)


Project dir: /content/drive/MyDrive/NMT_Project_Conv


## 2. Load Conversational Parallel Data

We use the **OpenSubtitles** en-hi parallel corpus (movie/TV subtitles) via Hugging Face `datasets`.
This is informal, dialogue-heavy text — a good proxy for the "conversational" domain gap AI4Bharat
identifies in their own `IN22-Conv` benchmark.

If the `open_subtitles` loader script is unavailable in your environment, the fallback cell below
pulls the same corpus via the OPUS-mirrored parquet config instead. Try the primary cell first.

In [ ]:
# Primary loader
try:
    raw = load_dataset("open_subtitles", lang1="en", lang2="hi")
    print(raw)
except Exception as e:
    print("Primary loader failed:", e)
    raw = None


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Primary loader failed: Dataset scripts are no longer supported, but found open_subtitles.py


In [ ]:
# Fallback loader (only runs if primary failed) — Helsinki-NLP OPUS mirror on the HF Hub
if raw is None:
    raw = load_dataset("Helsinki-NLP/opus-100", "en-hi")
    print(raw)
    print("NOTE: opus-100 is a general sampled OPUS mix, not subtitles-specific — "
          "domain purity will be weaker than open_subtitles. Prefer the primary loader if at all possible.")


DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 534319
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
NOTE: opus-100 is a general sampled OPUS mix, not subtitles-specific — domain purity will be weaker than open_subtitles. Prefer the primary loader if at all possible.


In [ ]:
# Inspect structure — adapt the extraction below if the fallback branch was used,
# since open_subtitles and opus-100 expose the parallel pair slightly differently.
split_name = list(raw.keys())[0]
sample = raw[split_name][0]
print(sample)


{'translation': {'en': 'Give shots of injections or pills, but he must be alright soon.', 'hi': 'सुई लगाओ या गोली खिलाओ लेकिन इसे जल्दी से ठीक करो.'}}


In [ ]:
def extract_pairs(dataset_split):
    """Normalize whichever loader ran into a flat list of (en, hi) tuples."""
    pairs = []
    for ex in dataset_split:
        # open_subtitles format: {'translation': {'en': ..., 'hi': ...}}
        # opus-100 format: {'translation': {'en': ..., 'hi': ...}}  (same shape, both use 'translation')
        tr = ex.get("translation", ex)
        en = tr.get("en", "").strip()
        hi = tr.get("hi", "").strip()
        if en and hi:
            pairs.append((en, hi))
    return pairs

all_pairs = extract_pairs(raw[split_name])
print("Total raw pairs:", len(all_pairs))
print(all_pairs[:3])


Total raw pairs: 2000
[('Give shots of injections or pills, but he must be alright soon.', 'सुई लगाओ या गोली खिलाओ लेकिन इसे जल्दी से ठीक करो.'), ('They said, “O Shuaib, we do not understand much of what you say, and we see that you are weak among us. Were it not for your tribe, we would have stoned you. You are of no value to us.”', 'और वह लोग कहने लगे ऐ शुएब जो बाते तुम कहते हो उनमें से अक्सर तो हमारी समझ ही में नहीं आयी और इसमें तो शक नहीं कि हम तुम्हें अपने लोगों में बहुत कमज़ोर समझते है और अगर तुम्हारा क़बीला न होता तो हम तुम को (कब का) संगसार कर चुके होते और तुम तो हम पर किसी तरह ग़ालिब नहीं आ सकते'), ('- Yeah.', '- हाँ.')]


## 3. Clean, Dedupe, Filter, Split

Subtitle corpora are noisy: lots of exact-duplicate lines ("Yes.", "No.", "What?"), misaligned pairs,
and very short/very long outliers. Basic filtering matters more here than for curated corpora.

In [ ]:
def clean_pairs(pairs, min_len=2, max_len=40, max_ratio=3.0):
    seen = set()
    cleaned = []
    for en, hi in pairs:
        key = (en, hi)
        if key in seen:
            continue
        seen.add(key)

        en_len = len(en.split())
        hi_len = len(hi.split())
        if en_len < min_len or hi_len < min_len:
            continue
        if en_len > max_len or hi_len > max_len:
            continue
        # guard against wildly misaligned length ratios (common subtitle alignment noise)
        ratio = max(en_len, hi_len) / max(1, min(en_len, hi_len))
        if ratio > max_ratio:
            continue
        # basic sanity: hi side should contain Devanagari characters
        if not re.search(r"[\u0900-\u097F]", hi):
            continue

        cleaned.append((en, hi))
    return cleaned

cleaned_pairs = clean_pairs(all_pairs)
print(f"Pairs before cleaning: {len(all_pairs)}")
print(f"Pairs after cleaning:  {len(cleaned_pairs)}")


Pairs before cleaning: 2000
Pairs after cleaning:  1488


In [ ]:
random.shuffle(cleaned_pairs)

n_total = len(cleaned_pairs)
n_train = min(TRAIN_SIZE, n_total - VAL_SIZE)
n_val = min(VAL_SIZE, n_total - n_train)

train_pairs = cleaned_pairs[:n_train]
val_pairs = cleaned_pairs[n_train:n_train + n_val]

print(f"Train pairs: {len(train_pairs)}")
print(f"Val pairs:   {len(val_pairs)}")

def pairs_to_dataset(pairs):
    return Dataset.from_dict({
        "translation": [{"en": en, "hi": hi} for en, hi in pairs]
    })

train_raw = pairs_to_dataset(train_pairs)
val_raw = pairs_to_dataset(val_pairs)


Train pairs: 488
Val pairs:   1000


In [ ]:
import huggingface_hub

In [ ]:
from huggingface_hub import login
login()

## 4. Preprocessing

Source text (encoder input) uses `is_target=False` (default) — the language-tag prefix is required.
**Target text (decoder labels) uses `is_target=True`** — no tag prefix, since this is what corrupted
the earlier experiment when it was left out.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
ip = IndicProcessor(inference=False)

def make_preprocess_function(tokenizer, ip, src_lang=SRC_LANG, tgt_lang=TGT_LANG,
                              max_length=MAX_TOKEN_LENGTH):
    def preprocess_function(examples):
        english = [ex["en"] for ex in examples["translation"]]
        hindi = [ex["hi"] for ex in examples["translation"]]

        # Source: default is_target=False -> language-tag prefix IS added (correct, required)
        source_texts = ip.preprocess_batch(english, src_lang=src_lang, tgt_lang=tgt_lang)
        # Target/labels: is_target=True -> NO tag prefix added (correct — this is the fix)
        target_texts = ip.preprocess_batch(hindi, src_lang=tgt_lang, tgt_lang=src_lang, is_target=True)

        source = tokenizer(source_texts, truncation=True, max_length=max_length, padding=False)
        with tokenizer.as_target_tokenizer():
            target = tokenizer(target_texts, truncation=True, max_length=max_length, padding=False)

        return {
            "input_ids": source["input_ids"],
            "attention_mask": source["attention_mask"],
            "labels": target["input_ids"],
        }
    return preprocess_function

preprocess_function = make_preprocess_function(tokenizer, ip)

# Quick sanity check on 2 examples before running over the full set
mini_check = train_raw.select(range(2)).map(preprocess_function, batched=True, batch_size=2,
                                             remove_columns=["translation"])
print(tokenizer.decode(mini_check[0]["labels"], skip_special_tokens=False))
print(tokenizer.decode(mini_check[1]["labels"], skip_special_tokens=False))
print("^ Confirm: Hindi text starts directly, no 'hin_Deva'/'eng_Latn'-looking tag tokens up front.")


Parameter 'function'=<function make_preprocess_function.<locals>.preprocess_function at 0x7c26b9ed0cc0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

फ़ाइल ( F )</s>
तुम से यह पूछने कोः</s>
^ Confirm: Hindi text starts directly, no 'hin_Deva'/'eng_Latn'-looking tag tokens up front.


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:4109: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
train_dataset = train_raw.map(
    preprocess_function, batched=True, batch_size=64, remove_columns=["translation"]
)
val_dataset = val_raw.map(
    preprocess_function, batched=True, batch_size=64, remove_columns=["translation"]
)

train_dataset.save_to_disk(TOKENIZED_TRAIN_PATH)
val_dataset.save_to_disk(TOKENIZED_VAL_PATH)

print(train_dataset)
print(val_dataset)


Map:   0%|          | 0/488 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/488 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 488
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})


## 5. Baseline Evaluation (Base Model, Zero-Shot)

Establish the base model's BLEU/chrF++ on this conversational held-out set **before** any fine-tuning.
This is the number LoRA needs to beat.

In [ ]:
def batch_translate(model, tokenizer, ip, sentences, src_lang=SRC_LANG, tgt_lang=TGT_LANG,
                     batch_size=16, num_beams=5, max_length=MAX_TOKEN_LENGTH):
    model.eval()
    outputs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        preproc = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)
        inputs = tokenizer(preproc, truncation=True, padding=True,
                            max_length=max_length, return_tensors="pt").to(model.device)
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                num_beams=num_beams,
                max_length=max_length,
                use_cache=True,
            )
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        postproc = ip.postprocess_batch(decoded, lang=tgt_lang)
        outputs.extend(postproc)
    return outputs


In [ ]:
from tqdm.auto import tqdm

def batch_translate(model, tokenizer, ip, sentences, src_lang=SRC_LANG, tgt_lang=TGT_LANG,
                     batch_size=16, num_beams=5, max_length=MAX_TOKEN_LENGTH, desc="Translating"):
    model.eval()
    outputs = []
    for i in tqdm(range(0, len(sentences), batch_size), desc=desc, unit="batch"):
        batch = sentences[i:i + batch_size]
        preproc = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)
        inputs = tokenizer(preproc, truncation=True, padding=True,
                            max_length=max_length, return_tensors="pt").to(model.device)
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                num_beams=num_beams,
                max_length=max_length,
                use_cache=True,
            )
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        postproc = ip.postprocess_batch(decoded, lang=tgt_lang)
        outputs.extend(postproc)
    return outputs

In [ ]:
!nvidia-smi

Sun Aug 23 09:57:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# num_beams=1 (greedy) is much faster on CPU; bump to 5 when running on GPU
NUM_BEAMS = 5 if device.type == "cuda" else 1
EVAL_BATCH_SIZE = 16 if device.type == "cuda" else 4

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL, trust_remote_code=True, torch_dtype=torch.float16
).to(device)

val_en = [ex["en"] for ex in val_raw["translation"]]
val_hi_ref = [ex["hi"] for ex in val_raw["translation"]]

t0 = time.time()
base_predictions = batch_translate(base_model, tokenizer, ip, val_en,
                                    batch_size=EVAL_BATCH_SIZE, num_beams=NUM_BEAMS)
print(f"Base model translation took {time.time() - t0:.1f}s for {len(val_en)} sentences")


Translating:   0%|          | 0/63 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [ ]:
base_bleu = sacrebleu.corpus_bleu(base_predictions, [val_hi_ref])
base_chrf = sacrebleu.corpus_chrf(base_predictions, [val_hi_ref], word_order=2)  # chrF++

print(f"Base BLEU:   {base_bleu.score:.2f}")
print(f"Base chrF++: {base_chrf.score:.2f}")


## 6. LoRA Fine-Tuning

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model,
    label_pad_token_id=-100,
    padding=True,
)

use_fp16 = device.type == "cuda"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.10,
    weight_decay=0.0,
    max_grad_norm=1.0,
    num_train_epochs=1,
    fp16=use_fp16,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=NUM_BEAMS,
    generation_max_length=MAX_TOKEN_LENGTH,
    logging_steps=50,
    report_to="none",
    label_names=["labels"],
    disable_tqdm=False,
)

print(training_args)


In [ ]:
trainer = Seq2SeqTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()


In [ ]:
best_checkpoint = trainer.state.best_model_checkpoint
print("Best checkpoint:", best_checkpoint)

summary = {
    "base_model": BASE_MODEL,
    "domain": "conversational (OpenSubtitles en-hi)",
    "train_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "epochs": training_args.num_train_epochs,
    "learning_rate": training_args.learning_rate,
    "per_device_train_batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "target_modules": lora_config.target_modules,
    "device": str(device),
    "num_beams": NUM_BEAMS,
    "best_checkpoint": best_checkpoint,
    "base_bleu": base_bleu.score,
    "base_chrf": base_chrf.score,
}

with open(os.path.join(OUTPUT_DIR, "conv_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


## 7. Evaluate Fine-Tuned Model

Reload the base model fresh + attach the saved LoRA adapter from the best checkpoint, so this
evaluation is independent of whatever is left in memory from training.

In [ ]:
eval_base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, trust_remote_code=True)
lora_eval_model = PeftModel.from_pretrained(eval_base_model, best_checkpoint).to(device)
lora_eval_model.eval()

t0 = time.time()
lora_predictions = batch_translate(lora_eval_model, tokenizer, ip, val_en,
                                    batch_size=EVAL_BATCH_SIZE, num_beams=NUM_BEAMS)
print(f"LoRA model translation took {time.time() - t0:.1f}s for {len(val_en)} sentences")


In [ ]:
lora_bleu = sacrebleu.corpus_bleu(lora_predictions, [val_hi_ref])
lora_chrf = sacrebleu.corpus_chrf(lora_predictions, [val_hi_ref], word_order=2)

print("=" * 60)
print(f"Base BLEU:   {base_bleu.score:.2f}   ->  LoRA BLEU:   {lora_bleu.score:.2f}")
print(f"Base chrF++: {base_chrf.score:.2f}   ->  LoRA chrF++: {lora_chrf.score:.2f}")
print("=" * 60)

summary["lora_bleu"] = lora_bleu.score
summary["lora_chrf"] = lora_chrf.score
with open(os.path.join(OUTPUT_DIR, "conv_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)


In [ ]:
# Qualitative spot-check
print("Sample translations (EN | Reference HI | Base | LoRA)\n")
for i in random.sample(range(len(val_en)), k=min(5, len(val_en))):
    print(f"EN:   {val_en[i]}")
    print(f"REF:  {val_hi_ref[i]}")
    print(f"BASE: {base_predictions[i]}")
    print(f"LORA: {lora_predictions[i]}")
    print("-" * 100)


## 8. Optional: Evaluate on Official IN22-Conv Benchmark

For a citable, third-party benchmark (rather than only your own held-out subtitle split), also
evaluate on AI4Bharat's official conversational benchmark. This is the strongest evidence for a
resume/portfolio writeup, since it's not a set you built or filtered yourself.

In [ ]:
try:
    in22_conv = load_dataset("ai4bharat/IN22-Conv")
    print(in22_conv)
except Exception as e:
    print("Could not load IN22-Conv directly:", e)
    print("If this fails, download it manually from: "
          "https://github.com/AI4Bharat/IndicTrans2 (see IN22 benchmark section) "
          "and adapt the loading cell to read the local files.")
    in22_conv = None


In [ ]:
if in22_conv is not None:
    # Adapt field names to whatever the actual IN22-Conv schema uses — inspect one example first
    split_name_conv = list(in22_conv.keys())[0]
    print(in22_conv[split_name_conv][0])


> Inspect the printed example above and adjust the extraction logic (English / Hindi field names)
> to match the actual IN22-Conv schema, then reuse `batch_translate` + `sacrebleu` exactly as in
> Section 5/7 to get base-vs-LoRA numbers on this official benchmark.

## Summary

- **Domain:** conversational English–Hindi (OpenSubtitles), chosen because AI4Bharat's own IN22-Conv
  benchmark indicates this is a genuine weak spot for the base model, unlike generic-domain text
  where the base model is already close to saturated.
- **Bug avoided by design:** target/label preprocessing always uses `is_target=True`, preventing the
  language-tag-prefix corruption that previously collapsed BLEU from ~25 to ~5 in an earlier general-domain
  experiment.
- **Result:** see `Base BLEU -> LoRA BLEU` / `Base chrF++ -> LoRA chrF++` printed above, and the full
  run config in `conv_summary.json`.
